# QTrans—平衡二分类公平嵌套调参

本 Notebook 诊断 QTrans 在 UCR Wafer 和 SECOM 上的训练不足或超参不匹配问题。调参过程不评估测试性能：

- UCR Wafer 代码只打开 `Wafer_TRAIN.txt`，完全不打开 `Wafer_TEST.txt`；
- SECOM 保留原有 5 个外层测试折，每个外折的调参只使用其余开发样本；
- 四个模型都有 8 个候选配置，使用相同内层分割和选择规则；
- 参数量差强制小于 1%；
- 依据 `平均验证 Macro-F1 - 0.25 × 标准差` 选择并冻结配置。

## 重要的发表边界

由于旧版 QTrans 已经查看过两个数据集的测试汇总结果，这次改进应视为「新一轮模型开发」，不能冒充为事前预注册试验。冻结新配置后，最强证据应来自尚未查看的新数据集或新外部留出集。

In [1]:
from pathlib import Path
import pandas as pd
import torch

from qcs_balanced_nested_tuning import (
    candidate_table, epoch_diagnostics, expected_jobs,
    finalize_nested_selection, nested_parameter_audit,
    run_nested_search, tuning_progress, tuning_summary,
)

pd.set_option('display.max_columns', 100)
PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'qcs_balanced_nested_tuning.py').exists():
    raise RuntimeError('请从 /root/xxx/autodl 目录打开本 Notebook')
UCR_DIR = PROJECT_DIR / 'data' / 'raw' / 'Wafer'
SECOM_DIR = PROJECT_DIR / 'data' / 'raw' / 'secom'
ARTIFACT_ROOT = PROJECT_DIR / 'artifacts' / 'balanced_binary_nested_tuning'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
display(pd.DataFrame([{'device': str(DEVICE), 'artifacts': str(ARTIFACT_ROOT)}]))

/root/xxx/autodl/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,device,artifacts
0,cuda,/root/xxx/autodl/artifacts/balanced_binary_nes...


## 1. 调参空间与公平性审计

候选空间是预先写死的，不会根据本次运行的分数动态追加候选。`epochs` 是最大训练轮数，检查点仍由验证交叉熵选择。

In [2]:
display(candidate_table())
display(pd.DataFrame([{
    'dataset': 'ucr_wafer', 'expected_jobs': expected_jobs('ucr_wafer'),
}, {
    'dataset': 'secom', 'expected_jobs': expected_jobs('secom'),
}]))
display(nested_parameter_audit(152))
display(nested_parameter_audit(40))

,candidate_id,epochs,patience,learning_rate,weight_decay,dropout,label_smoothing,quantum_depth,quantum_init_scale,quantum_pre_norm,quantum_trainable_stabilizers,quantum_attention_temperature,quantum_residual_scale,quantum_lr_multiplier
0,0,120,20,0.0005,0.0300,0.25,0.08,2,0.10,False,False,1.00,1.00,1.00
1,1,200,35,0.0002,0.0100,0.15,0.03,2,0.05,True,True,1.00,0.25,0.50
2,2,160,30,0.0003,0.0010,0.10,0.00,3,0.03,True,True,0.75,0.50,0.25
3,3,100,18,0.0008,0.0100,0.10,0.03,2,0.15,True,True,1.50,0.50,2.00
4,4,240,45,0.0001,0.0010,0.05,0.00,3,0.05,False,True,0.50,0.25,1.00
5,5,160,25,0.0005,0.0001,0.15,0.02,2,0.03,True,True,1.00,1.00,2.00
6,6,80,15,0.0010,0.0010,0.05,0.00,3,0.10,True,True,1.50,1.00,0.50
7,7,200,35,0.0003,0.0300,0.30,0.05,2,0.15,False,True,0.75,0.50,0.25


,dataset,expected_jobs
0,ucr_wafer,192
1,secom,480


,candidate_id,model,parameters,relative_to_quantum
0,0,quantum_transformer,7202,0.000000
1,0,tiny_transformer,7258,0.007776
2,0,mlp_mixer,7257,0.007637
3,0,cnn_token_mixer,7194,-0.001111
4,1,quantum_transformer,7204,0.000000
5,1,tiny_transformer,7258,0.007496
6,1,mlp_mixer,7257,0.007357
7,1,cnn_token_mixer,7194,-0.001388
8,2,quantum_transformer,7244,0.000000
9,2,tiny_transformer,7258,0.001933


,candidate_id,model,parameters,relative_to_quantum
0,0,quantum_transformer,7090,0.000000
1,0,tiny_transformer,7146,0.007898
2,0,mlp_mixer,7061,-0.004090
3,0,cnn_token_mixer,7082,-0.001128
4,1,quantum_transformer,7092,0.000000
5,1,tiny_transformer,7146,0.007614
6,1,mlp_mixer,7061,-0.004371
7,1,cnn_token_mixer,7082,-0.001410
8,2,quantum_transformer,7132,0.000000
9,2,tiny_transformer,7146,0.001963


## 2. UCR Wafer：官方 TRAIN 内部嵌套调参

194 个平衡开发样本进行 3 折×2次重复内层验证。共 `4模型 × 8候选 × 6内折 = 192` 个任务。

In [3]:
ucr_progress = tuning_progress(ARTIFACT_ROOT, 'ucr_wafer')
display(ucr_progress.groupby('model')['complete'].agg(['sum', 'count']))
print(f"UCR Wafer 已完成 {int(ucr_progress.complete.sum())}/{len(ucr_progress)}")

,sum,count
model,,
cnn_token_mixer,0,48
mlp_mixer,0,48
quantum_transformer,0,48
tiny_transformer,0,48


UCR Wafer 已完成 0/192


`MAX_JOBS_UCR=None` 表示完成所有剩余任务。首次可设为 `1` 做环境检查，之后恢复 `None`。

In [ ]:
MAX_JOBS_UCR = None
ucr_results = run_nested_search(
    dataset='ucr_wafer', data_dir=UCR_DIR, artifact_dir=ARTIFACT_ROOT,
    balance_seed=2026, inner_seed=8192,
    max_jobs=MAX_JOBS_UCR, device=DEVICE,
)
print(f'UCR Wafer 当前结果 {len(ucr_results)}/192')
display(tuning_summary(ucr_results).groupby('model').head(3))

[ucr_wafer] outer=-1, inner=0, candidate=0, model=quantum_transformer, device=cuda
[ucr_wafer] outer=-1, inner=0, candidate=0, model=tiny_transformer, device=cuda
[ucr_wafer] outer=-1, inner=0, candidate=0, model=mlp_mixer, device=cuda
[ucr_wafer] outer=-1, inner=0, candidate=0, model=cnn_token_mixer, device=cuda
[ucr_wafer] outer=-1, inner=0, candidate=1, model=quantum_transformer, device=cuda


In [ ]:
ucr_progress = tuning_progress(ARTIFACT_ROOT, 'ucr_wafer')
if not ucr_progress.complete.all():
    raise RuntimeError(f'UCR Wafer 未完成：{int(ucr_progress.complete.sum())}/192')
ucr_summary, ucr_frozen = finalize_nested_selection(
    ARTIFACT_ROOT, 'ucr_wafer'
)
display(pd.DataFrame(ucr_frozen['selections']))
display(epoch_diagnostics(ucr_results).query("model == 'quantum_transformer'"))

## 3. SECOM：5个外折各自进行内层调参

每个外层测试折都不进入对应调参过程。共 `5外折 × 4模型 × 8候选 × 3内折 = 480` 个任务。缺失值填充、ANOVA 特征选择和标准化每次只在内层训练部分拟合。

In [ ]:
secom_progress = tuning_progress(ARTIFACT_ROOT, 'secom')
display(secom_progress.groupby(['outer_fold', 'model'])['complete'].agg(['sum', 'count']))
print(f"SECOM 已完成 {int(secom_progress.complete.sum())}/{len(secom_progress)}")

In [ ]:
MAX_JOBS_SECOM = None
secom_results = run_nested_search(
    dataset='secom', data_dir=SECOM_DIR, artifact_dir=ARTIFACT_ROOT,
    balance_seed=2026, outer_seed=4096, inner_seed=8192,
    max_jobs=MAX_JOBS_SECOM, device=DEVICE,
)
print(f'SECOM 当前结果 {len(secom_results)}/480')
display(tuning_summary(secom_results).groupby(['outer_fold', 'model']).head(2))

In [ ]:
secom_progress = tuning_progress(ARTIFACT_ROOT, 'secom')
if not secom_progress.complete.all():
    raise RuntimeError(f'SECOM 未完成：{int(secom_progress.complete.sum())}/480')
secom_summary, secom_frozen = finalize_nested_selection(
    ARTIFACT_ROOT, 'secom'
)
display(pd.DataFrame(secom_frozen['selections']))
display(epoch_diagnostics(secom_results).query("model == 'quantum_transformer'"))

## 4. 怎样判断是否为 epoch 问题

- 如果高分候选的 `best_epoch_fraction_mean` 接近 1，且 `near_budget_end_rate` 很高，原有轮数可能不足；
- 如果最佳 epoch 远小于最大轮数，增加 epoch 不会解决问题，重点应转向学习率、梯度尺度、归一化和残差强度；
- 如果 QTrans 在多个内折选中不同候选，且标准差仍很大，问题更可能是优化不稳定而不是单纯的轮数不足；
- 任何新配置都要等完成整个等预算搜索后才冻结，不得根据部分任务提前挑选。

## 输出位置

```text
artifacts/balanced_binary_nested_tuning/
├── ucr_wafer/
│   ├── validation_results.csv
│   ├── validation_summary.csv
│   └── frozen_nested_selection.json
└── secom/
    ├── validation_results.csv
    ├── validation_summary.csv
    └── frozen_nested_selection.json
```

这个 Notebook 不生成任何测试集分数。冻结结果完成后，再另建一个只读冻结配置的正式评估 Notebook。